In [3]:
import os 
api_key=os.getenv("SER_API")

if api_key is None:
    print("no API")
else:
    print("i find my key")

i find my key


## OpenRouter Connection

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    model="openai/gpt-4o-mini",
    temperature=0
)

response = llm.invoke("Hello")

print(response.content)

Hello! How can I assist you today?


## translate input users query  to Compilar arxiv query

In [36]:
#from langchain.prompts import PromptTemplate

from langchain_core.prompts import PromptTemplate

template = """
You are an arXiv query optimization system.

Your task is to convert user requests into concise and effective arXiv search queries.

Rules:
1. Use only concise technical keywords.
2. Avoid unnecessary phrases like:
   "research papers", "studies about", "articles about".
3. Expand abbreviations into standard technical terms when beneficial for retrieval.
4. Use Boolean operators (AND, OR) only if necessary.
5. Focus on retrieval quality for arXiv.
6. Return ONLY the final query.
7. Keep the query short and search-engine friendly.

User Request:
{user_input}

Optimized arXiv Query:"""

query_prompt = PromptTemplate.from_template(template)

# Function to translate query
def generate_arxiv_query(user_input):
    chain = query_prompt | llm
    response = chain.invoke({"user_input": user_input})
    return response.content.strip()

old one:
template = """
You are an academic retrieval query optimizer.

Your task is to transform the user's request into an optimized arXiv search query.

Rules:
1. Convert the user intent into academic search terminology.
2. Use concise research keywords.
3. Use Boolean operators (AND, OR) when useful.
4. Expand concepts using standard research terms when necessary.
5. Focus on maximizing retrieval relevance.
6. Return ONLY the final arXiv query.

User Request:
{user_input}

Optimized arXiv Query:
"""

Prefix, Description, Example

`ti:`, Title, Searches for the word only in the title of the research paper.

`abs:`, Abstract, Searches the abstract of the research (the part that describes the patent or innovation).

`au:`, Author, Searches for a specific author.

`cat:`, Subject Category, Searches within a specific category (e.g., cs.AI for artificial intelligence).

`all:`, All fields, Searches in all fields (default).


In [37]:
querys = generate_arxiv_query("I want papers about using deep learning for image classification in medical imaging")
test_query= generate_arxiv_query("Jeffrey Hinton's Deep Learning Research")
print(querys)
print(test_query)

deep learning AND image classification AND medical imaging
Hinton Deep Learning


## LLM Query Optimizer → arXiv Search (old)

## Structured Schema Design

In [38]:
import arxiv
def to_schema(result):
    return {  #Clean Retrieval Output (Good Practice)
            "paper_id": result.entry_id,
            "title": result.title,
            "year": result.published.year,
            "published": str(result.published),
            "authors": [author.name for author in result.authors],
            "summary": result.summary.replace("\n", " ").strip(),
            "url": result.pdf_url,
        }
def search_arxiv(query):

    client = arxiv.Client() ###

    search = arxiv.Search(
        query=query,
        max_results=2,
        sort_by=arxiv.SortCriterion.Relevance
    )

    results = []
    for result in client.results(search):
        results.append(to_schema(result))
    return results


In [39]:
#only test
query = generate_arxiv_query(test_query)
results = search_arxiv(query)
print(query)

Hinton Deep Learning


In [33]:
#only test
q2 = "robotics AND AI"

results = search_arxiv(q2)

print(len(results))

2


In [40]:
#only test 
for r in results:
    print("=" * 40)
    print(r["title"])

Learn to Accumulate Evidence from All Training Samples: Theory and Practice
The Modern Mathematics of Deep Learning


In [ ]:
# only test
test_query2 = "stil papers about UAS systems"

query3 = generate_arxiv_query(test_query2)

print("QUERY:", query3)

results = search_arxiv(query3) ### this

print("RESULT COUNT:", len(results))

for r in results:
    print(r["title"])



QUERY: UAS systems
RESULT COUNT: 2
Counter-Unmanned Aircraft System(s) (C-UAS): State of the Art, Challenges and Future Trends
Symmetries and periodic orbits in simple hybrid Routhian systems


by llm:

- strengths
- weaknesses
- key takeaway
- critical analysis
- keywords

In [42]:
import json

qa = generate_arxiv_query(
    "deep learning medical imaging"
)

results = search_arxiv(qa)
print(json.dumps(results, indent=4))

[
    {
        "paper_id": "http://arxiv.org/abs/2302.04143v2",
        "title": "Predicting Thrombectomy Recanalization from CT Imaging Using Deep Learning Models",
        "year": 2023,
        "published": "2023-02-08 15:41:21+00:00",
        "authors": [
            "Haoyue Zhang",
            "Jennifer S. Polson",
            "Eric J. Yang",
            "Kambiz Nael",
            "William Speier",
            "Corey W. Arnold"
        ],
        "summary": "For acute ischemic stroke (AIS) patients with large vessel occlusions, clinicians must decide if the benefit of mechanical thrombectomy (MTB) outweighs the risks and potential complications following an invasive procedure. Pre-treatment computed tomography (CT) and angiography (CTA) are widely used to characterize occlusions in the brain vasculature. If a patient is deemed eligible, a modified treatment in cerebral ischemia (mTICI) score will be used to grade how well blood flow is reestablished throughout and following the MT

## Embeddings